# External Data, Visualization & Modular Code

# Part 1 — Read an external data source

In [1]:
## Importing Libraries
import pandas as pd
from pathlib import Path
print(Path.cwd())

/workspaces/STAT556-Homework-2/notebooks


In [2]:
## Data Directory and Relative Path

DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

In [3]:
## Reading Raw Data
ExpressMart_df = pd.read_csv(RAW_DIR / "ExpressMartData.TOT.csv")

# Inspect Shape, Columns, Dtypes, and Missingness.

In [4]:
# Inspecting Shape
ExpressMart_df.shape

(185950, 8)

In [5]:
# Inspecting Columns
ExpressMart_df.columns

Index(['Order ID', 'Product', 'Quantity Ordered', 'Price Each', 'Order Date',
       'Purchase Address', 'Time', 'Sales'],
      dtype='str')

In [6]:
# Inspecting Data Types
ExpressMart_df.dtypes

Order ID              int64
Product                 str
Quantity Ordered      int64
Price Each          float64
Order Date            int64
Purchase Address        str
Time                float64
Sales               float64
dtype: object

In [7]:
## Checking Missingness
ExpressMart_df.isna().sum()

Order ID            0
Product             0
Quantity Ordered    0
Price Each          0
Order Date          0
Purchase Address    0
Time                0
Sales               0
dtype: int64

In [8]:
ExpressMart_df.head(10)

,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,Time,Sales
0,295665,Macbook Pro Laptop,1,1700.00,43829,"136 Church St, New York City, NY 10001",0.000694,1700.00
1,295666,LG Washing Machine,1,600.00,43828,"562 2nd St, New York City, NY 10001",0.293750,600.00
2,295667,USB-C Charging Cable,1,11.95,43811,"277 Main St, New York City, NY 10001",0.764583,11.95
3,295668,27in FHD Monitor,1,149.99,43821,"410 6th St, San Francisco, CA 94016",0.634028,149.99
4,295669,USB-C Charging Cable,1,11.95,43817,"43 Hill St, Atlanta, GA 30301",0.526389,11.95
5,295670,AA Batteries (4-pack),1,3.84,43830,"200 Jefferson St, New York City, NY 10001",0.956944,3.84
6,295671,USB-C Charging Cable,1,11.95,43815,"928 12th St, Portland, OR 97035",0.631944,11.95
7,295672,USB-C Charging Cable,2,11.95,43812,"813 Hickory St, Dallas, TX 75001",0.395139,23.90
8,295673,Bose SoundSport Headphones,1,99.99,43814,"718 Wilson St, Dallas, TX 75001",0.976389,99.99
9,295674,AAA Batteries (4-pack),4,2.99,43827,"77 7th St, Dallas, TX 75001",0.493750,11.96


In [9]:
ExpressMart_df.describe()

,Order ID,Quantity Ordered,Price Each,Order Date,Time,Sales
count,185950.000000,185950.000000,185950.000000,185950.000000,185950.000000,185950.000000
mean,230417.569379,1.124383,184.399735,43664.291923,0.621028,185.490917
std,51512.737110,0.442793,332.731330,107.159471,0.226111,332.919771
min,141234.000000,1.000000,2.990000,43466.000000,0.000000,2.990000
25%,185831.250000,1.000000,11.950000,43571.000000,0.470139,11.950000
50%,230367.500000,1.000000,14.950000,43663.000000,0.632639,14.950000
75%,275035.750000,1.000000,150.000000,43764.000000,0.806250,150.000000
max,319670.000000,9.000000,1700.000000,43831.000000,0.999306,3400.000000


# Part 2 — Modular cleaning

In [10]:
import sys
sys.path.append("..")

from src.clean import clean_column_names, add_location, drop_exact_duplicates, parse_order_datetime, filter_year, clean_raw_data

Changing Column Names

In [11]:
ExpressClean_df = clean_column_names(ExpressMart_df)

In [12]:
ExpressClean_df.head()

,order_id,product,quantity_ordered,price_each,order_date,purchase_address,time,sales
0,295665,Macbook Pro Laptop,1,1700.00,43829,"136 Church St, New York City, NY 10001",0.000694,1700.00
1,295666,LG Washing Machine,1,600.00,43828,"562 2nd St, New York City, NY 10001",0.293750,600.00
2,295667,USB-C Charging Cable,1,11.95,43811,"277 Main St, New York City, NY 10001",0.764583,11.95
3,295668,27in FHD Monitor,1,149.99,43821,"410 6th St, San Francisco, CA 94016",0.634028,149.99
4,295669,USB-C Charging Cable,1,11.95,43817,"43 Hill St, Atlanta, GA 30301",0.526389,11.95


Separating Street, City And State from Purchase Address

In [13]:
df_location = add_location(ExpressClean_df)
df_location.head()

,order_id,product,quantity_ordered,price_each,order_date,purchase_address,time,sales,city,state,city_state
0,295665,Macbook Pro Laptop,1,1700.00,43829,"136 Church St, New York City, NY 10001",0.000694,1700.00,New York City,NY,"New York City, NY"
1,295666,LG Washing Machine,1,600.00,43828,"562 2nd St, New York City, NY 10001",0.293750,600.00,New York City,NY,"New York City, NY"
2,295667,USB-C Charging Cable,1,11.95,43811,"277 Main St, New York City, NY 10001",0.764583,11.95,New York City,NY,"New York City, NY"
3,295668,27in FHD Monitor,1,149.99,43821,"410 6th St, San Francisco, CA 94016",0.634028,149.99,San Francisco,CA,"San Francisco, CA"
4,295669,USB-C Charging Cable,1,11.95,43817,"43 Hill St, Atlanta, GA 30301",0.526389,11.95,Atlanta,GA,"Atlanta, GA"


Duplicated Cells Verification

In [14]:
print(df_location.duplicated().sum())

264


In [15]:
print(df_location["order_id"].duplicated().sum())

7513


In [16]:
df_location[df_location.duplicated(keep=False)].sort_values("order_id").head(6)

,order_id,product,quantity_ordered,price_each,order_date,purchase_address,time,sales,city,state,city_state
137664,142071,AA Batteries (4-pack),1,3.84,43482,"131 2nd St, Boston, MA 02215",0.959722,3.84,Boston,MA,"Boston, MA"
137665,142071,AA Batteries (4-pack),1,3.84,43482,"131 2nd St, Boston, MA 02215",0.959722,3.84,Boston,MA,"Boston, MA"
140898,145143,Lightning Charging Cable,1,14.95,43471,"182 Jefferson St, San Francisco, CA 94016",0.125694,14.95,San Francisco,CA,"San Francisco, CA"
140899,145143,Lightning Charging Cable,1,14.95,43471,"182 Jefferson St, San Francisco, CA 94016",0.125694,14.95,San Francisco,CA,"San Francisco, CA"
142577,146765,Google Phone,1,600.00,43486,"918 Highland St, New York City, NY 10001",0.474306,600.00,New York City,NY,"New York City, NY"
142578,146765,Google Phone,1,600.00,43486,"918 Highland St, New York City, NY 10001",0.474306,600.00,New York City,NY,"New York City, NY"


In [17]:
df_nodup = drop_exact_duplicates(df_location)
print("before:", len(df_location), "after:", len(df_nodup), "removed:", len(df_location) - len(df_nodup))

before: 185950 after: 185686 removed: 264


Parse Order Datetime Piece 1

In [18]:
df_nodup[["order_date", "time"]].head(3)

,order_date,time
0,43829,0.000694
1,43828,0.293750
2,43811,0.764583


In [19]:
pd.to_datetime(43829, unit="D", origin="1899-12-30")

Timestamp('2019-12-30 00:00:00')

In [20]:
df_datetime = parse_order_datetime(df_nodup)
df_datetime.head()

,order_id,product,quantity_ordered,price_each,purchase_address,sales,city,state,city_state,order_datetime,month,hour
0,295665,Macbook Pro Laptop,1,1700.00,"136 Church St, New York City, NY 10001",1700.00,New York City,NY,"New York City, NY",2019-12-30 00:01:00,12,0
1,295666,LG Washing Machine,1,600.00,"562 2nd St, New York City, NY 10001",600.00,New York City,NY,"New York City, NY",2019-12-29 07:03:00,12,7
2,295667,USB-C Charging Cable,1,11.95,"277 Main St, New York City, NY 10001",11.95,New York City,NY,"New York City, NY",2019-12-12 18:21:00,12,18
3,295668,27in FHD Monitor,1,149.99,"410 6th St, San Francisco, CA 94016",149.99,San Francisco,CA,"San Francisco, CA",2019-12-22 15:13:00,12,15
4,295669,USB-C Charging Cable,1,11.95,"43 Hill St, Atlanta, GA 30301",11.95,Atlanta,GA,"Atlanta, GA",2019-12-18 12:38:00,12,12


Year Filter

In [21]:
df_2019 = filter_year(df_datetime)
print("before:", len(df_datetime), "after:", len(df_2019), "removed:", len(df_datetime) - len(df_2019))
print(df_2019["order_datetime"].min(), df_2019["order_datetime"].max())

before: 185686 after: 185652 removed: 34
2019-01-01 03:07:00 2019-12-31 23:53:00


In [22]:
clean_df = clean_raw_data(ExpressMart_df)

In [23]:
clean_df.shape

(185652, 12)

In [24]:
clean_df.columns

Index(['order_id', 'product', 'quantity_ordered', 'price_each',
       'purchase_address', 'sales', 'order_datetime', 'month', 'hour', 'city',
       'state', 'city_state'],
      dtype='str')

In [25]:
ExpressMart_df.shape

(185950, 8)